# **＜重要＞本章で使用するファイル**

- fitlife_members.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

# Menu.75 教師なし学習とクラスタリングを理解しよう

データの基本情報を確認する

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("fitlife_members.csv")

print(df.head())
print(df.info())

基本統計量を確認する

In [ ]:
print(df.describe())

カテゴリ変数の分布を確認する

In [ ]:
cols = ["preferred_workout_type", "preferred_time", "goal"]
for col in cols:
  print(df[col].value_counts())
  print()

クラスタリングのための前処理をする

In [ ]:
# 特徴量の選択（user_idを除く）
X = df.drop(columns=["user_id"])

# カテゴリ変数をダミー変数に変換
X = pd.get_dummies(X, drop_first=True)

# 標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("前処理後のデータ形状:", X_scaled.shape)
print(X_scaled.columns)

前処理済みデータを保存する

In [ ]:
# 前処理済みデータを保存
X_scaled.to_csv("X_scaled.csv", index=False)

# Menu.76 隠れたパターンを発見！k-meansクラスタリングを実装しよう

前処理済みデータを読み込む

In [ ]:
import pandas as pd

X_scaled = pd.read_csv("X_scaled.csv")

k-meansモデルを構築する

In [ ]:
from sklearn.cluster import KMeans

# k-meansクラスタリング (k=3)
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_scaled)
cluster_labels = kmeans.labels_

print(cluster_labels[:10]) # 最初の10件のクラスタラベルを表示

各クラスタの人数を確認する

In [ ]:
print("各クラスタの人数:")
print(pd.Series(cluster_labels).value_counts())

n_initパラメータを使う

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)
cluster_labels = kmeans.labels_

print("各クラスタの人数:")
print(pd.Series(cluster_labels).value_counts())

# Menu.77 グループ数はいくつが良い？エルボー法でクラスタ数を決めよう

エルボー法を実装する

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# 前処理済みデータ読み込み
X_scaled = pd.read_csv("X_scaled.csv")

# k=2-10でinertia計算
inertias = []
K_range = range(2, 11)

for k in K_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  kmeans.fit(X_scaled)
  inertias.append(kmeans.inertia_)

# プロット
plt.plot(K_range, inertias, marker="o", linewidth=2)
plt.xlabel("k", fontsize=12)
plt.ylabel("Inertia", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Menu.78 もう1つの視点！シルエット分析でクラスタ数を評価しよう

シルエットスコアを計算する

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 前処理済みデータ読み込み
X_scaled = pd.read_csv("X_scaled.csv")

# k=2-10でシルエットスコア計算
K_range = range(2, 11)

for k in K_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  kmeans.fit(X_scaled)
  labels = kmeans.labels_
  score = silhouette_score(X_scaled, labels)
  print("k =", k, score)

# Menu.79 グループの特徴は？クラスタ中心を取得して解釈しよう

クラスタ中心を可視化する

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# 前処理済みデータから特徴量名を取得
X_scaled = pd.read_csv("X_scaled.csv")
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_scaled)

feature_names = X_scaled.columns

# クラスタ中心
centers = kmeans.cluster_centers_
centers_df = pd.DataFrame(centers, columns=feature_names)

# ヒートマップ
plt.figure(figsize=(12, 4))
sns.heatmap(centers_df, cmap="RdBu_r", center=0, annot=True, fmt=".2f")
plt.ylabel("cluster", fontsize=12)
plt.show()

# Menu.80 別のアプローチ！階層的クラスタリングとデンドログラムを学ぼう

Ward法で階層的クラスタリングを実装する

In [ ]:
import pandas as pd
from scipy.cluster.hierarchy import linkage

# 前処理済みデータ読み込み
X_scaled = pd.read_csv("X_scaled.csv")

# Ward法で階層的クラスタリング
linkage_matrix = linkage(X_scaled, method="ward")

# linkage行列の中身を確認
print("linkage行列の形状:", linkage_matrix.shape)
print("最初の5行（最初に結合されたペア）:")
print(linkage_matrix[:5])
print("最後の2行（最後に結合されたペア）:")
print(linkage_matrix[-2:])

デンドログラムを描画する

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

# デンドログラム描画
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.xlabel("Cluster", fontsize=12)
plt.ylabel("Distance", fontsize=12)
plt.title("dendrogram (Ward)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()